# Advanced Statistical Analysis & Hypothesis Testing

**Project:** Employee Performance Statistical Analysis

This notebook performs:
- Shapiro-Wilk and Kolmogorov-Smirnov normality tests
- Independent two-sample t-test
- 95% confidence interval
- Mann-Whitney U test
- One-Way ANOVA
- Tukey HSD post-hoc analysis
- Two-Way ANOVA
- Automatic saving of statistical results to CSV files

**Significance level (α): 0.05**


In [ ]:
# Cell 1: Install required libraries
# Run this cell once if the libraries are not installed.

!pip install pandas numpy scipy statsmodels matplotlib seaborn openpyxl


In [ ]:
# Cell 2: Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

from pathlib import Path

ALPHA = 0.05

# Create output folder automatically
Path("results").mkdir(exist_ok=True)

print("Libraries imported successfully!")


In [ ]:
# Cell 3: Load dataset

# The CSV file should be inside the data folder.
df = pd.read_csv("data/employee_performance.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()


In [ ]:
# Cell 4: Basic dataset information

print("Columns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())


In [ ]:
# Cell 5: Descriptive statistics

print("Overall descriptive statistics:")
display(df.describe())

print("\nPerformance by Department:")
department_summary = (
    df.groupby("Department")["Performance_Score"]
    .agg(["count", "mean", "std", "min", "max"])
)

display(department_summary)


In [ ]:
# Cell 6: Visualization - Performance by Department

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x="Department",
    y="Performance_Score"
)

plt.title("Performance Score by Department")
plt.xlabel("Department")
plt.ylabel("Performance Score")
plt.show()


## 1. Normality Tests

### Hypotheses

**Shapiro-Wilk**
- H₀: The data follows a normal distribution.
- H₁: The data does not follow a normal distribution.

**Kolmogorov-Smirnov**
- H₀: The sample follows the specified normal distribution.
- H₁: The sample does not follow the specified normal distribution.

Decision rule:
- p < 0.05 → Reject H₀
- p ≥ 0.05 → Fail to reject H₀


In [ ]:
# Cell 7: Shapiro-Wilk normality test

performance = df["Performance_Score"].dropna()

shapiro_stat, shapiro_p = stats.shapiro(performance)

print("Shapiro-Wilk Normality Test")
print("---------------------------")
print(f"Statistic: {shapiro_stat:.4f}")
print(f"p-value: {shapiro_p:.4f}")

if shapiro_p < ALPHA:
    print("Conclusion: Reject H0")
    print("The data is not normally distributed.")
else:
    print("Conclusion: Fail to reject H0")
    print("There is no significant evidence of non-normality.")


In [ ]:
# Cell 8: Kolmogorov-Smirnov normality test

# Standardize the sample before comparing it with a standard normal distribution.
standardized = (
    performance - performance.mean()
) / performance.std()

ks_stat, ks_p = stats.kstest(
    standardized,
    "norm"
)

print("Kolmogorov-Smirnov Normality Test")
print("--------------------------------")
print(f"Statistic: {ks_stat:.4f}")
print(f"p-value: {ks_p:.4f}")

if ks_p < ALPHA:
    print("Conclusion: Reject H0")
else:
    print("Conclusion: Fail to reject H0")


In [ ]:
# Cell 9: Save normality test results

normality_results = pd.DataFrame({
    "Test": ["Shapiro-Wilk", "Kolmogorov-Smirnov"],
    "Statistic": [shapiro_stat, ks_stat],
    "p_value": [shapiro_p, ks_p],
    "Alpha": [ALPHA, ALPHA]
})

display(normality_results)

normality_results.to_csv(
    "results/normality_tests.csv",
    index=False
)

print("normality_tests.csv created successfully!")


## 2. Two-Sample Hypothesis Tests

We compare **Performance Score** between:
- Online training
- Classroom training

### Independent t-test hypotheses

- H₀: The two group means are equal.
- H₁: The two group means are different.

### Mann-Whitney U hypotheses

- H₀: The two groups have the same distribution.
- H₁: The two groups have different distributions.


In [ ]:
# Cell 10: Prepare the two groups

online = df.loc[
    df["Training_Method"] == "Online",
    "Performance_Score"
].dropna()

classroom = df.loc[
    df["Training_Method"] == "Classroom",
    "Performance_Score"
].dropna()

print("Online group size:", len(online))
print("Classroom group size:", len(classroom))

print("\nOnline mean:", round(online.mean(), 2))
print("Classroom mean:", round(classroom.mean(), 2))


In [ ]:
# Cell 11: Independent two-sample t-test

# Welch's t-test is used because it does not require equal variances.
t_stat, t_p = stats.ttest_ind(
    online,
    classroom,
    equal_var=False
)

print("Independent Two-Sample t-test")
print("-----------------------------")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {t_p:.4f}")

if t_p < ALPHA:
    print("Conclusion: Reject H0")
    print("There is a statistically significant difference between the group means.")
else:
    print("Conclusion: Fail to reject H0")
    print("There is no statistically significant difference between the group means.")


In [ ]:
# Cell 12: 95% confidence interval for difference in means

mean_online = online.mean()
mean_classroom = classroom.mean()

difference = mean_online - mean_classroom

se = np.sqrt(
    online.var(ddof=1) / len(online)
    + classroom.var(ddof=1) / len(classroom)
)

df_welch = (
    (
        online.var(ddof=1) / len(online)
        + classroom.var(ddof=1) / len(classroom)
    ) ** 2
    /
    (
        (online.var(ddof=1) / len(online)) ** 2 / (len(online) - 1)
        +
        (classroom.var(ddof=1) / len(classroom)) ** 2 / (len(classroom) - 1)
    )
)

critical_value = stats.t.ppf(
    1 - ALPHA / 2,
    df_welch
)

margin = critical_value * se

ci_lower = difference - margin
ci_upper = difference + margin

print("Mean difference (Online - Classroom):", round(difference, 4))
print("95% Confidence Interval:")
print("Lower:", round(ci_lower, 4))
print("Upper:", round(ci_upper, 4))


In [ ]:
# Cell 13: Mann-Whitney U test

u_stat, u_p = stats.mannwhitneyu(
    online,
    classroom,
    alternative="two-sided"
)

print("Mann-Whitney U Test")
print("-------------------")
print(f"U-statistic: {u_stat:.4f}")
print(f"p-value: {u_p:.4f}")

if u_p < ALPHA:
    print("Conclusion: Reject H0")
    print("The two groups are statistically different.")
else:
    print("Conclusion: Fail to reject H0")
    print("No statistically significant difference was detected.")


In [ ]:
# Cell 14: Save two-sample test results

hypothesis_results = pd.DataFrame({
    "Test": ["Independent t-test", "Mann-Whitney U"],
    "Statistic": [t_stat, u_stat],
    "p_value": [t_p, u_p],
    "Alpha": [ALPHA, ALPHA]
})

display(hypothesis_results)

hypothesis_results.to_csv(
    "results/hypothesis_tests.csv",
    index=False
)

print("hypothesis_tests.csv created successfully!")


## 3. One-Way ANOVA

We test whether the average **Performance Score** differs among departments.

### Hypotheses

- H₀: All department means are equal.
- H₁: At least one department mean is different.

Decision rule:
- p < 0.05 → Reject H₀
- p ≥ 0.05 → Fail to reject H₀


In [ ]:
# Cell 15: One-Way ANOVA

groups = [
    group["Performance_Score"].dropna()
    for _, group in df.groupby("Department")
]

f_stat, anova_p = stats.f_oneway(*groups)

print("One-Way ANOVA")
print("-------------")
print(f"F-statistic: {f_stat:.4f}")
print(f"p-value: {anova_p:.4f}")

if anova_p < ALPHA:
    print("Conclusion: Reject H0")
    print("At least one department has a different mean.")
else:
    print("Conclusion: Fail to reject H0")
    print("No significant difference between department means.")


In [ ]:
# Cell 16: Tukey HSD post-hoc analysis

tukey = pairwise_tukeyhsd(
    endog=df["Performance_Score"],
    groups=df["Department"],
    alpha=ALPHA
)

print(tukey)


In [ ]:
# Cell 17: Save Tukey HSD results

tukey_results = pd.DataFrame(
    data=tukey._results_table.data[1:],
    columns=tukey._results_table.data[0]
)

display(tukey_results)

tukey_results.to_csv(
    "results/tukey_results.csv",
    index=False
)

print("tukey_results.csv created successfully!")


## 4. Two-Way ANOVA

We examine:
1. The effect of **Department**
2. The effect of **Training Method**
3. The **Department × Training Method interaction**

### Hypotheses

**Department**
- H₀: Department has no effect on performance.
- H₁: Department has an effect on performance.

**Training Method**
- H₀: Training method has no effect on performance.
- H₁: Training method has an effect on performance.

**Interaction**
- H₀: There is no Department × Training Method interaction.
- H₁: There is a Department × Training Method interaction.


In [ ]:
# Cell 18: Two-Way ANOVA

model = ols(
    "Performance_Score ~ C(Department) + "
    "C(Training_Method) + "
    "C(Department):C(Training_Method)",
    data=df
).fit()

two_way_anova = sm.stats.anova_lm(
    model,
    typ=2
)

display(two_way_anova)


In [ ]:
# Cell 19: Interpret Two-Way ANOVA

print("Two-Way ANOVA Interpretation")
print("============================")

for factor in two_way_anova.index:
    p_value = two_way_anova.loc[factor, "PR(>F)"]

    if p_value < ALPHA:
        print(f"{factor}: SIGNIFICANT (p = {p_value:.4f})")
    else:
        print(f"{factor}: NOT SIGNIFICANT (p = {p_value:.4f})")


In [ ]:
# Cell 20: Save ANOVA results

two_way_anova.to_csv(
    "results/anova_results.csv"
)

print("anova_results.csv created successfully!")


## 5. Final Results Summary

All statistical results are saved automatically in the `results` folder.

Generated files:
- `normality_tests.csv`
- `hypothesis_tests.csv`
- `anova_results.csv`
- `tukey_results.csv`


In [ ]:
# Cell 21: Verify generated result files

print("Generated result files:")
print("-----------------------")

for file in sorted(Path("results").glob("*.csv")):
    print("✓", file)


In [ ]:
# Cell 22: Compact final summary

print("FINAL STATISTICAL SUMMARY")
print("=========================")

print(f"Shapiro-Wilk p-value: {shapiro_p:.4f}")
print(f"Kolmogorov-Smirnov p-value: {ks_p:.4f}")
print(f"Independent t-test p-value: {t_p:.4f}")
print(f"Mann-Whitney U p-value: {u_p:.4f}")
print(f"One-Way ANOVA p-value: {anova_p:.4f}")

print("\nTwo-Way ANOVA:")
display(two_way_anova)
